# Balance de Potencia y Disponibilidad

Este notebook implementa un **simulador interactivo de un enlace OFDM** en el que la calidad del canal ya no se fija directamente mediante la SNR, sino que se obtiene a partir de un **balance de potencia**.

La cadena modelada es:

1. Definicion de la potencia transmitida y ganancias de antena  
2. Calculo de las perdidas del enlace  
3. Obtencion de la **SNR nominal** a partir del balance de enlace  
4. Simulacion del sistema OFDM sobre un canal AWGN o con fading  
5. Estimacion de la **BER**  
6. Calculo de la probabilidad de recepcion correcta de un paquete

El objetivo didactico es relacionar dos niveles del sistema:

- el nivel **fisico del enlace**: potencia, ganancias, perdidas, distancia y frecuencia
- el nivel **digital**: constelacion, BER, recepcion correcta de paquetes y disponibilidad

---

## Parametros del notebook

### Parametros del sistema

**Ptx (dBm)**  
Potencia de transmision del emisor. Un valor mas alto aumenta la potencia recibida y, en general, mejora la SNR.

**Gtx / Grx (dBi)**  
Ganancias de las antenas transmisora y receptora. Ganancias mayores incrementan la potencia util recibida.

**Pkt (B)**  
Longitud del paquete en bytes. En general, un paquete mas corto tiene más probabilidad de recibirse sin errores.

**Freq. (MHz)**  
Frecuencia de portadora de la señal a transmitir. Cuanto más alta, mayor será la atenuación debida a la propagación en espacio libre.


---

### Parametros del canal

**Loss mode**  
Permite elegir entre dos formas de modelar las perdidas del enlace:

- `manual`: se introduce directamente la perdida total en dB
- `fspl`: la perdida se calcula a partir de la distancia y la frecuencia usando propagacion en espacio libre: $L[dB] = 32.44 + 20.0 · log_{10}(d_{km}) + 20.0 · log_{10}(f_{Mhz})$

**Loss (dB)**  
Perdida del enlace cuando se trabaja en modo manual.

**Dist (m)** y **Freq (MHz)**  
Parametros usados cuando se trabaja en modo `fspl`.

**BW (Hz)**  
Ancho de banda del receptor, necesario para estimar la potencia de ruido.

**NF (dB)**  
Figura de ruido del receptor. Valores mayores degradan la SNR.

**Lmisc (dB)**  
Perdidas adicionales del sistema no incluidas en la propagacion.

**Fading**  
Selecciona si el canal es:

- `none`: sin fading
- `slow`: fading Rayleigh lento
- `fast`: fading Rayleigh rapido

**n_taps**  
Numero de trayectorias del canal Rayleigh.

---

### Parametros OFDM

**Nsub**  
Numero de subportadoras OFDM.

**CP**  
Longitud del prefijo ciclico.

**M-QAM**  
Orden de la constelacion. El notebook soporta `4`, `16` y `64`.

**Frames**  
Numero de tramas OFDM usadas en cada iteracion del simulador.

**FEC ON**, **FEC n**, **FEC k**  
Activan la codificacion de canal y fijan su tasa.

---

## Salidas del simulador

En ejecucion, el notebook muestra:

- constelacion transmitida y recibida
- SNR instantanea en las ultimas iteraciones
- BER instantanea en las ultimas iteraciones
- disponibilidad por bloques de iteraciones
- estado de recepcion correcta o incorrecta del paquete

La **disponibilidad** se interpreta como el porcentaje de paquetes correctamente recibidos dentro de cada bloque de observacion.

---

## Uso basico

1. Ajustar los parametros del enlace, del canal y del sistema OFDM  
2. Pulsar **Start sim** para lanzar la simulacion periodica  
3. Observar como cambian constelacion, SNR, BER y disponibilidad  
4. Pulsar **Stop sim** para detener la simulacion


## Arranque en Colab

Estas celdas preparan el entorno para ejecutar el notebook en **Google Colab** usando **solo** el paquete compilado en `Engine/`.

Para esta version cliente, la carpeta `Engine/smartgrids_sim/` debe contener ya los binarios Linux (`*.so`). No se utiliza `src/`.

Si tu carpeta en Drive no esta en la ruta indicada, modifica la variable `PROJECT_DIR` en la siguiente celda.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/SGA_Simulations/3_Sistema'
%cd $PROJECT_DIR

!pip install -q numpy scipy matplotlib ipywidgets
engine_pkg = Path('Engine/smartgrids_sim')
if not engine_pkg.exists():
    raise FileNotFoundError('No existe Engine/smartgrids_sim. Debes subir la version compilada para Linux.')
so_files = sorted(engine_pkg.glob('*.so'))
pyd_files = sorted(engine_pkg.glob('*.pyd'))
print('Engine package:', engine_pkg.resolve())
print('SO files:', [p.name for p in so_files])
print('PYD files:', [p.name for p in pyd_files])
if pyd_files:
    raise RuntimeError('Se han detectado binarios Windows (.pyd). En Colab necesitas binarios Linux (.so).')
if not so_files:
    raise RuntimeError('No se han encontrado ficheros .so en Engine/smartgrids_sim.')
print((engine_pkg / '__init__.py').read_text(encoding='utf-8'))


In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
package_root = project_root / 'Engine'
helper_root = project_root / 'Helpers'

for path in [package_root, helper_root]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print('Project root:', project_root)
print('Active package root:', package_root)
print('Helper root:', helper_root)


In [ ]:
from link_balance_helper import *


## Notas

- El notebook delega la logica interactiva en `Helpers/link_balance_helper.py`.
- El motor de simulacion sigue viviendo en `src/smartgrids_sim/`.
- La SNR mostrada en el historico es la **instantanea**, afectada por la realizacion concreta del canal.
- La disponibilidad se calcula por bloques independientes de iteraciones, no de forma acumulada desde el arranque.


## Preguntas

1. Fije una constelacion **64-QAM** y compare dos casos: `none` fading y `slow` fading.  
   ¿Que observa en la disponibilidad y en la estabilidad de la recepcion de paquetes?

2. En el caso `slow` fading con **64-QAM** y **sin FEC**, observe si puede darse una situacion en la que la **BER siga siendo baja** pero la **disponibilidad** caiga de forma apreciable.  
   ¿Como explicaria esa aparente contradiccion?

3. ¿Que efecto tienen la **potencia transmitida** y las **ganancias de antena** sobre la calidad del enlace?

4. ¿Por que un paquete mas largo tiene menos probabilidad de recibirse correctamente cuando la BER no es nula?

5. Compare los casos `none`, `slow` y `fast` fading.  
   ¿En cual esperaria una variacion temporal mas rapida del enlace?

6. Mantenga `slow` fading y **64-QAM**, y active el **FEC**.  
   ¿Que ocurre con la disponibilidad y con la recepcion de paquetes?  
   ¿Por que puede recuperarse el rendimiento aunque la condicion del canal siga siendo exigente?

7. En esa misma situacion, pruebe una alternativa distinta: desactive el FEC y baje la constelacion de **64-QAM** a **16-QAM**.  
   ¿Se recupera tambien la disponibilidad? Compare esta solucion con la de usar FEC.

8. Compare las dos estrategias de recuperacion del rendimiento con $L = 110 dB$ y fading lento:

- **64-QAM + FEC**
- **16-QAM sin FEC**

   Observe que ambas ofrecen una disponibilidad alta, ¿cual proporciona mejor **tasa de transmision util**?
   Justifique la respuesta a partir de los bits por simbolo y de la redundancia introducida por el FEC.


<details>
<summary>Ver soluciones</summary>

### 1

Con `none` fading la disponibilidad suele ser mas estable y mas alta. Con `slow` fading aparecen intervalos en los que el canal empeora y la recepcion de paquetes deja de ser tan fiable.

---

### 2

No hay una contradiccion real. Una BER baja significa que el error medio por bit puede seguir siendo pequeno, pero un paquete completo exige que no falle NINGÚN bit. En constelaciones de orden alto, pequenos aumentos del error pueden penalizar mucho la disponibilidad de paquetes.

---

### 3

Aumentar la potencia transmitida o las ganancias de antena incrementa la potencia recibida y, por tanto, suele mejorar la calidad del enlace y reducir la BER.

---

### 4

Cuantos mas bits tenga el paquete, mas oportunidades hay de que alguno llegue con error. Por eso la probabilidad de paquete correcto disminuye al crecer su longitud.

---

### 5

El caso `fast` fading produce cambios mas rapidos entre iteraciones. `slow` fading varia mas lentamente y `none` elimina ese efecto.

---

### 6

El FEC introduce redundancia y permite corregir parte de los errores. Por eso puede recuperar la disponibilidad incluso cuando el canal sigue siendo exigente: la constelacion no mejora magicamente, pero el receptor tolera mejor los errores que aparecen.

---

### 7

Si se baja de 64-QAM a 16-QAM, los puntos de constelacion quedan mas separados y la deteccion se vuelve mas robusta. Esto tambien puede recuperar la disponibilidad, aunque a costa de reducir los bits transmitidos por simbolo.

---

### 8

La comparacion debe hacerse en terminos de tasa util. `64-QAM` transmite mas bits por simbolo que `16-QAM`, pero el FEC consume parte de esa ventaja porque introduce redundancia. Si ambas configuraciones alcanzan disponibilidad alta, la mejor sera la que mantenga mayor tasa util efectiva, no solo mejor BER o mejor disponibilidad. El uso del FEC supone una disminución de la tasa de transmisión por un factor $R_c=k/n$. Para $n=3$ y $k=1$, mientras que la diferencia de tasa de transmisión entre 64-QAM y 16-QAM es de $6/4=1.5$. Es decir, 64-QAM es 1.5 veces más rápido que 16-QAM. En términos de tasa de transmisión, es más eficiente utilizar 16-QAM sin FEC que 64-QAM con FEC (para estos valores de $n$ y $k$, y dado que los valores de disponibilidad son similares)
</details>

---
